# Protocol 6 — Intracranial EEG: All-or-None Ignition Dynamics

Simulates predictions Pred 6.A–Pred 6.D from :

* **Pred 6.A** — Frontoparietal high-gamma shows bimodal (not graded) distribution at threshold
* **Pred 6.B** — Bimodality is specific to frontoparietal electrodes; occipital units are graded
* **Pred 6.C** — AC1 of pre-ignition high-gamma increases before detected stimuli *(bifurcation criterion: distinguishes APGI from standard GWT)*
* **Pred 6.D** — Near-threshold stimuli produce stable intermediate states > 100 ms

> Pred 6.C is the critical distinguishing prediction. Standard GWT predicts threshold-crossing without precursor dynamics. APGI predicts critical slowing indexed by AC1 increase.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import ks_2samp

: 

## 1 — Protocol 6 parameters

In [ ]:
N_NEAR_THRESHOLD = 150  # 50% of 300 trials
AC1_BINS = 10  # 10 x 50 ms windows in 500 ms pre-ignition epoch
rng = np.random.default_rng(9)

# Frontoparietal: bimodal at threshold (detected = high state, missed = low state)
hg_detected = rng.normal(0.72, 0.08, N_NEAR_THRESHOLD // 2)
hg_missed = rng.normal(0.21, 0.07, N_NEAR_THRESHOLD // 2)
hg_fp = np.concatenate([hg_detected, hg_missed])

# Occipital: unimodal graded response (Pred 6.B)
hg_occ = rng.normal(0.45, 0.18, N_NEAR_THRESHOLD)

## 2 — Prediction 6.A/Pred 6.B: Bimodal vs graded distributions

In [ ]:
from scipy.stats import ks_2samp

# Hartigan dip test approximation: KS distance between sample and a normal fit
from scipy.stats import norm as spnorm

# KS test: fp distribution vs best-fit normal (bimodal deviates from normal)
mean_fp, std_fp = hg_fp.mean(), hg_fp.std()
_, p_fp = ks_2samp(hg_fp, spnorm.rvs(mean_fp, std_fp, size=2000, random_state=42))
mean_oc, std_oc = hg_occ.mean(), hg_occ.std()
_, p_occ = ks_2samp(hg_occ, spnorm.rvs(mean_oc, std_oc, size=2000, random_state=42))
print(f"Frontoparietal KS vs normal: p = {p_fp:.4f}  (small p = bimodal)")
print(f"Occipital KS vs normal:       p = {p_occ:.4f}  (large p = graded)")

## 3 — Prediction 6.C: AC1 pre-ignition slowing (bifurcation criterion)

In [ ]:
time_bins = np.linspace(-500, 0, AC1_BINS)
ac1_detected = np.clip(
    0.20 + 0.035 * np.arange(AC1_BINS) + rng.normal(0, 0.025, AC1_BINS), 0, 1
)
ac1_missed = np.clip(0.22 + rng.normal(0, 0.025, AC1_BINS), 0, 1)

# Kendall tau for trend in detected trials
from scipy.stats import kendalltau

tau, p_tau = kendalltau(np.arange(AC1_BINS), ac1_detected)
print(
    f"AC1 trend (detected): Kendall τ = {tau:.3f}, p = {p_tau:.4f}  (Pred 6.C: expect τ > 0.3, p < 0.05)"
)

## 4 — Visualise all four predictions

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(16, 4))

# Prediction 6.A: frontoparietal bimodal
ax = axes[0]
ax.hist(
    hg_detected,
    bins=18,
    color="#2166ac",
    alpha=0.75,
    edgecolor="white",
    label="Detected",
    density=True,
)
ax.hist(
    hg_missed,
    bins=18,
    color="#d6604d",
    alpha=0.75,
    edgecolor="white",
    label="Non-detected",
    density=True,
)
ax.set_title("Pred 6.A — Frontoparietal\nbimodal")
ax.set_xlabel("High-gamma (a.u.)")
ax.legend(fontsize=7)
ax.spines[["top", "right"]].set_visible(False)

# Prediction 6.B: occipital graded vs frontoparietal
ax = axes[1]
ax.hist(
    hg_fp,
    bins=20,
    color="#2166ac",
    alpha=0.70,
    edgecolor="white",
    label="Frontoparietal",
    density=True,
)
ax.hist(
    hg_occ,
    bins=20,
    color="#AAAAAA",
    alpha=0.65,
    edgecolor="white",
    label="Occipital",
    density=True,
)
ax.set_title("Pred 6.B — Regional\nspecificity")
ax.set_xlabel("High-gamma (a.u.)")
ax.legend(fontsize=7)
ax.spines[["top", "right"]].set_visible(False)

# Prediction 6.C: AC1 pre-ignition slowing
ax = axes[2]
ax.plot(time_bins, ac1_detected, "o-", color="#2166ac", lw=1.8, ms=5, label="Detected")
ax.plot(
    time_bins,
    ac1_missed,
    "s--",
    color="#d6604d",
    lw=1.5,
    ms=4,
    alpha=0.8,
    label="Non-detected",
)
ax.set_title("Pred 6.C — AC1 pre-ignition\nslowing (bifurcation)")
ax.set_xlabel("Time before ignition (ms)")
ax.set_ylabel("AC1")
ax.legend(fontsize=7)
ax.spines[["top", "right"]].set_visible(False)

# Prediction 6.D: stable intermediate states
prop_intermediate = 0.18  # > 15% per protocol prediction
ax = axes[3]
ax.bar(
    ["Intermediate state\n(> 100 ms)"],
    [prop_intermediate],
    color="#9966FF",
    alpha=0.85,
    edgecolor="white",
    width=0.4,
)
ax.axhline(0.15, ls="--", lw=1, color="k", alpha=0.5, label="15% threshold")
ax.set_title("Pred 6.D — Stable intermediate\nstates")
ax.set_ylabel("Proportion of trials")
ax.set_ylim(0, 0.4)
ax.legend(fontsize=8)
ax.spines[["top", "right"]].set_visible(False)

fig.suptitle(
    "Protocol 6 — iEEG All-or-None Ignition Dynamics (Pred 6.A–Pred 6.D)", fontsize=12
)
plt.tight_layout()
plt.show()

## 5 — Why Prediction 6.C matters: APGI vs. Global Workspace Theory

| Prediction | Standard GWT | APGI |
|---|---|---|
| Bistable firing at threshold | Yes | Yes |
| Regional specificity (frontoparietal) | Yes | Yes |
| **AC1 pre-ignition increase (critical slowing)** | **No** | **Yes** |
| Stable intermediate states > 100 ms | Possible | Yes (bifurcation) |

P6c is the *bifurcation falsification criterion*. Failure of Pred 6.c does not falsify APGI threshold-crossing per se, but eliminates the bifurcation interpretation of ignition dynamics.